# P-Task10: Async Data Sync Dashboard

**Problem:** data team ko aisa tool chahiye jo **teen mock sources se ek saath** data khinche, results merge
kare aur combined summary dikhaye. Tool virtual environment ke andar chale aur saari credentials `.env` file mein
hon (code mein nahi).

**Concepts used:** `asyncio` (`async def`, `await`, `asyncio.run()`, `asyncio.gather()`), `threading` + `Lock`,
`multiprocessing.Pool`, GIL, `venv`, `.env` + `python-dotenv`, `pip freeze`.

## 1. Environment setup (terminal mein, ek baar)
```bash
python -m venv venv                  # virtual environment banao
source venv/bin/activate             # Windows: venv\Scripts\activate
pip install -r requirements.txt      # python-dotenv
cp .env.example .env                 # phir .env mein apni keys likho
pip freeze > requirements.txt        # (jab naye packages lagen to)
deactivate                           # venv se bahar
```
**Kyun venv?** Har project ke apne packages hote hain, isliye ek project ka version doosre ko kharab nahi karta.

**Notebook mein note:** Jupyter mein pehle se event loop chal raha hota hai, isliye wahan `asyncio.run()` ki
jagah seedha `await` likhte hain. Script file mein `asyncio.run(main())` use hota hai.

## 2. `.env` se credentials lena
Keys code mein nahi likhi. `python-dotenv` `.env` file ko padh kar environment variables bana deta hai.
Neeche sirf masked (chhupi hui) key print ki hai.

In [1]:
import asyncio
import os
import random
import threading
import time
from multiprocessing import Pool

from dotenv import load_dotenv

import workers            # CPU function (multiprocessing ke liye)

load_dotenv(".env")       # .env na ho to environment variables hi use honge

KEYS = {
    "students": os.getenv("STUDENT_API_KEY"),
    "fees": os.getenv("FEES_API_KEY"),
    "courses": os.getenv("COURSE_API_KEY"),
}
TIMEOUT = float(os.getenv("REQUEST_TIMEOUT", "3"))

missing = [name for name, key in KEYS.items() if not key]
if missing:
    raise SystemExit(f".env mein keys nahi mili: {missing}. Pehle .env.example se .env banao.")

def mask(key):
    return key[:3] + "*" * (len(key) - 3)

{name: mask(key) for name, key in KEYS.items()}, "timeout =", TIMEOUT

({'students': 'stu***********', 'fees': 'fee***********', 'courses': 'crs***********'}, 'timeout =', 3.0)

## 3. Teen mock sources
Asli API nahi hai, isliye `asyncio.sleep()` se network ka intezar simulate kiya. Har source apni key check karti hai.

In [2]:
async def fetch_students(key):
    await asyncio.sleep(1.0)            # network ka intezar
    return [{"id": 1, "name": "Ayesha"}, {"id": 2, "name": "Bilal"}, {"id": 3, "name": "Fatima"}]


async def fetch_fees(key):
    await asyncio.sleep(1.5)
    return [{"student_id": 1, "paid": 50000, "due": 0},
            {"student_id": 2, "paid": 30000, "due": 20000},
            {"student_id": 3, "paid": 0, "due": 50000}]


async def fetch_courses(key):
    await asyncio.sleep(0.8)
    return [{"student_id": 1, "courses": 5}, {"student_id": 2, "courses": 4}, {"student_id": 3, "courses": 6}]


async def safe_fetch(name, coro_func):
    """Ek source fail ho to baqi na rukein, isliye try/except aur timeout."""
    try:
        return name, await asyncio.wait_for(coro_func(KEYS[name]), timeout=TIMEOUT)
    except asyncio.TimeoutError:
        return name, "TIMEOUT"
    except Exception as e:
        return name, f"ERROR: {e}"

## 4. Pehle ek ek karke (sequential), phir ek saath (`gather`)
`await` ke dauran Python doosra kaam shuru kar deta hai, isliye teeno ka intezar saath saath hota hai.

In [3]:
async def run_sequential():
    start = time.perf_counter()
    results = {}
    for name, func in [("students", fetch_students), ("fees", fetch_fees), ("courses", fetch_courses)]:
        key, data = await safe_fetch(name, func)
        results[key] = data
    return results, time.perf_counter() - start


async def run_concurrent():
    start = time.perf_counter()
    pairs = await asyncio.gather(
        safe_fetch("students", fetch_students),
        safe_fetch("fees", fetch_fees),
        safe_fetch("courses", fetch_courses),
    )
    return dict(pairs), time.perf_counter() - start


_, t_seq = await run_sequential()
results, t_async = await run_concurrent()

print(f"Sequential : {t_seq:.2f} sec")
print(f"gather()   : {t_async:.2f} sec   (sab se slow source jitna, 1.5 sec)")

Sequential : 3.30 sec
gather()   : 1.50 sec   (sab se slow source jitna, 1.5 sec)


## 5. Merge aur combined summary

In [4]:
def merge(results):
    fees = {f["student_id"]: f for f in results["fees"]}
    courses = {c["student_id"]: c for c in results["courses"]}
    return [
        {"id": s["id"], "name": s["name"], "paid": fees[s["id"]]["paid"],
         "due": fees[s["id"]]["due"], "courses": courses[s["id"]]["courses"]}
        for s in results["students"]
    ]


merged = merge(results)

print("=" * 52)
print("STUDENT SYNC DASHBOARD")
print("=" * 52)
print(f"{'ID':<4}{'Name':<10}{'Paid':>9}{'Due':>9}{'Courses':>9}")
for row in merged:
    print(f"{row['id']:<4}{row['name']:<10}{row['paid']:>9,}{row['due']:>9,}{row['courses']:>9}")
print("-" * 52)
print("Total students :", len(merged))
print("Total paid     : Rs", f"{sum(r['paid'] for r in merged):,}")
print("Total due      : Rs", f"{sum(r['due'] for r in merged):,}")
print("Defaulters     :", [r['name'] for r in merged if r['due'] > 0])
print(f"Avg courses    : {sum(r['courses'] for r in merged) / len(merged):.1f}")

STUDENT SYNC DASHBOARD
ID  Name           Paid      Due  Courses
1   Ayesha       50,000        0        5
2   Bilal        30,000   20,000        4
3   Fatima            0   50,000        6
----------------------------------------------------
Total students : 3
Total paid     : Rs 80,000
Total due      : Rs 70,000
Defaulters     : ['Bilal', 'Fatima']
Avg courses    : 5.0


## 6. Ek source fail ho jaye to? (timeout test)
Courses wali source ko jaan-boojh kar 5 sec ka kiya (timeout 3 sec hai).

In [5]:
async def slow_courses(key):
    await asyncio.sleep(5)
    return []

pairs = await asyncio.gather(
    safe_fetch("students", fetch_students),
    safe_fetch("fees", fetch_fees),
    safe_fetch("courses", slow_courses),
)
{name: (data if isinstance(data, str) else f"{len(data)} rows") for name, data in pairs}

{'students': '3 rows', 'fees': '3 rows', 'courses': 'TIMEOUT'}

Baqi do sources ka data aa gaya aur teesri `TIMEOUT` ho gayi, poora program nahi ruka.

## 7. Threads, race condition aur Lock
Do thread ek hi counter badhayein to bina lock ke jama galat ho sakta hai (race condition).
`time.sleep(0)` se thread ko switch hone ka mauka diya taake masla saaf dikhe.

In [6]:
counter = {"value": 0}
lock = threading.Lock()


def add_without_lock(times):
    for _ in range(times):
        current = counter["value"]
        time.sleep(0)                      # yahan doosra thread beech mein aa sakta hai
        counter["value"] = current + 1


def add_with_lock(times):
    for _ in range(times):
        with lock:                         # ek waqt mein sirf ek thread
            current = counter["value"]
            time.sleep(0)
            counter["value"] = current + 1


def run_threads(target):
    counter["value"] = 0
    threads = [threading.Thread(target=target, args=(2000,)) for _ in range(4)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return counter["value"]


print("Bina lock (expected 8000):", run_threads(add_without_lock))
print("Lock ke saath            :", run_threads(add_with_lock))

Bina lock (expected 8000): 2000
Lock ke saath            : 8000


## 8. Multiprocessing aur GIL
**GIL** (Global Interpreter Lock): Python mein ek waqt mein sirf ek thread Python code chala sakta hai.
Isliye CPU wale kaam mein threads tez nahi karte. Ye kaam alag **processes** se tez hota hai.

| | Best kis ke liye |
|---|---|
| Threads | file / network wait (IO), chhote kaam |
| Async | bohat saari network requests ek saath |
| Processes | bhaari calculation (CPU) |

In [7]:
numbers = [2_000_000] * 4

start = time.perf_counter()
one_by_one = [workers.heavy_sum(n) for n in numbers]
t_normal = time.perf_counter() - start

start = time.perf_counter()
with Pool(processes=4) as pool:
    in_parallel = pool.map(workers.heavy_sum, numbers)
t_pool = time.perf_counter() - start

print(f"Normal loop : {t_normal:.2f} sec")
print(f"Pool (4)    : {t_pool:.2f} sec")
print("Result same?:", one_by_one == in_parallel)

Normal loop : 0.42 sec
Pool (4)    : 0.24 sec
Result same?: True


## Conclusion
Teeno mock sources `asyncio.gather()` se ek saath chali aur 3.3 sec ke bajaye lagbhag 1.5 sec mein data aa gaya.
Credentials `.env` mein hain, code mein nahi, aur project venv ke andar chalta hai. Saath mein threads ki race
condition (Lock se theek) aur multiprocessing se CPU kaam tez karna bhi dekha.